# D123 — Shopping Cart with Pure Functions

A functional version of the shopping-cart example.

The cart data is immutable. Every change returns a new cart.

## What is a pure function?

A pure function:

- produces the same output for the same input
- does not change its input
- does not change outside state

Pure functions are easy to unit test.

## Immutable cart data

- One item is a tuple of `(key, value)` pairs.
- A cart is a tuple of items.
- Tuples cannot be changed in place.
- Cart functions return new tuples.

In [ ]:
keyboard = (
    ("id", 1),
    ("name", "Keyboard"),
    ("qty", 2),
    ("unit_price", 1500),
)

cart = (keyboard,)

## Small item helpers

Keep data creation and reading in small functions.

In [ ]:
def create_cart_item(item_id, name, qty, unit_price):
    if qty <= 0:
        raise ValueError("Quantity must be positive")
    if unit_price < 0:
        raise ValueError("Unit price cannot be negative")
    return (("id", item_id), ("name", name),
            ("qty", qty), ("unit_price", unit_price))

def item_value(item, key):
    return dict(item)[key]

In [ ]:
def item_amount(item):
    return item_value(item, "qty") * item_value(item, "unit_price")

mouse = create_cart_item(2, "Mouse", 1, 500)
assert item_value(mouse, "name") == "Mouse"
assert item_amount(mouse) == 500

## Initialize a cart

An empty tuple represents an empty cart.

In [ ]:
def initialize_cart():
    return ()

cart = initialize_cart()
assert cart == ()

## Calculate cart state

The function returns `(items_count, amount)` without storing or changing state.

In [ ]:
def calculate_cart_total(cart):
    items_count = sum(item_value(x, "qty") for x in cart)
    amount = sum(item_amount(x) for x in cart)
    return items_count, amount

assert calculate_cart_total(()) == (0, 0)

## Replace selected item values

This helper creates a new item instead of changing the old tuple.

In [ ]:
def replace_item(item, qty=None, unit_price=None):
    new_qty = item_value(item, "qty") if qty is None else qty
    new_price = item_value(item, "unit_price") if unit_price is None else unit_price
    return create_cart_item(
        item_value(item, "id"), item_value(item, "name"),
        new_qty, new_price
    )

## Add an item

A new ID is appended. An existing ID receives additional quantity.

In [ ]:
def add_item_to_cart(cart, new_item):
    new_id = item_value(new_item, "id")
    if not any(item_value(x, "id") == new_id for x in cart):
        return cart + (new_item,)
    return tuple(
        replace_item(x, item_value(x, "qty") + item_value(new_item, "qty"))
        if item_value(x, "id") == new_id else x for x in cart
    )

In [ ]:
cart = initialize_cart()
new_cart = add_item_to_cart(cart, keyboard)

assert cart == ()
assert calculate_cart_total(new_cart) == (2, 3000)

## Remove an item

Filtering creates a new tuple without the selected ID.

In [ ]:
def remove_item_from_cart(cart, item_id):
    if not any(item_value(x, "id") == item_id for x in cart):
        raise KeyError("Item not found")
    return tuple(x for x in cart if item_value(x, "id") != item_id)

assert remove_item_from_cart((keyboard,), 1) == ()

## Empty a cart

The function returns a new empty tuple.

In [ ]:
def empty_cart(cart):
    return ()

assert empty_cart((keyboard, mouse)) == ()

## Update quantity or price

The matching item is replaced; all other item tuples are reused.

In [ ]:
def update_cart(cart, item_id, qty=None, unit_price=None):
    if qty is None and unit_price is None:
        raise ValueError("Provide quantity or unit price")
    if not any(item_value(x, "id") == item_id for x in cart):
        raise KeyError("Item not found")
    return tuple(
        replace_item(x, qty, unit_price)
        if item_value(x, "id") == item_id else x for x in cart
    )

In [ ]:
cart = (keyboard, mouse)
updated = update_cart(cart, 1, qty=1, unit_price=1200)

assert calculate_cart_total(cart) == (3, 3500)
assert calculate_cart_total(updated) == (2, 1700)

## Test class and setup

`setUp` creates fresh immutable test data before every test.

In [ ]:
import unittest

class TestShoppingCart(unittest.TestCase):
    def setUp(self):
        self.keyboard = create_cart_item(1, "Keyboard", 2, 1500)
        self.mouse = create_cart_item(2, "Mouse", 1, 500)
        self.cart = initialize_cart()

    def tearDown(self):
        self.cart = None

## Initialization and add tests

In [ ]:
def test_initialize_cart(self):
    self.assertEqual(self.cart, ())
    self.assertEqual(calculate_cart_total(self.cart), (0, 0))

def test_add_item(self):
    result = add_item_to_cart(self.cart, self.keyboard)
    self.assertEqual(calculate_cart_total(result), (2, 3000))

TestShoppingCart.test_initialize_cart = test_initialize_cart
TestShoppingCart.test_add_item = test_add_item

In [ ]:
def test_add_same_item_combines_quantity(self):
    cart = add_item_to_cart(self.cart, self.keyboard)
    one_more = create_cart_item(1, "Keyboard", 1, 1500)
    result = add_item_to_cart(cart, one_more)
    self.assertEqual(calculate_cart_total(result), (3, 4500))

TestShoppingCart.test_add_same_item_combines_quantity = (
    test_add_same_item_combines_quantity
)

## Remove and empty tests

In [ ]:
def test_remove_item(self):
    cart = (self.keyboard, self.mouse)
    result = remove_item_from_cart(cart, 1)
    self.assertEqual(calculate_cart_total(result), (1, 500))

def test_empty_cart(self):
    result = empty_cart((self.keyboard, self.mouse))
    self.assertEqual(result, ())

TestShoppingCart.test_remove_item = test_remove_item
TestShoppingCart.test_empty_cart = test_empty_cart

## Update tests

In [ ]:
def test_update_quantity(self):
    result = update_cart((self.keyboard,), 1, qty=3)
    self.assertEqual(calculate_cart_total(result), (3, 4500))

def test_update_price(self):
    result = update_cart((self.keyboard,), 1, unit_price=1200)
    self.assertEqual(calculate_cart_total(result), (2, 2400))

TestShoppingCart.test_update_quantity = test_update_quantity
TestShoppingCart.test_update_price = test_update_price

## Immutability test

The original cart must remain unchanged after an operation.

In [ ]:
def test_update_does_not_change_original(self):
    original = (self.keyboard,)
    updated = update_cart(original, 1, qty=5)
    self.assertEqual(calculate_cart_total(original), (2, 3000))
    self.assertEqual(calculate_cart_total(updated), (5, 7500))

TestShoppingCart.test_update_does_not_change_original = (
    test_update_does_not_change_original
)

## Error tests

In [ ]:
def test_invalid_item_values(self):
    with self.assertRaises(ValueError):
        create_cart_item(1, "Keyboard", 0, 1500)
    with self.assertRaises(ValueError):
        create_cart_item(1, "Keyboard", 1, -1)

TestShoppingCart.test_invalid_item_values = test_invalid_item_values

In [ ]:
def test_unknown_item_errors(self):
    with self.assertRaises(KeyError):
        remove_item_from_cart(self.cart, 99)
    with self.assertRaises(KeyError):
        update_cart(self.cart, 99, qty=2)

TestShoppingCart.test_unknown_item_errors = test_unknown_item_errors

## Build a test suite

The suite collects every `test_` method from the test class.

In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(
    TestShoppingCart
)

result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()
print("Tests run:", result.testsRun)

## Functional style versus class style

| Pure-function cart | Class-based cart |
|---|---|
| data and functions are separate | data and methods are together |
| returns a new cart | changes the cart object |
| original input stays unchanged | object maintains current state |
| tests compare input and output | tests also inspect object state |

## Key lessons

- Immutable data cannot be accidentally changed in place.
- Pure functions make inputs and outputs visible.
- Each cart operation returns a new cart.
- `setUp` provides fresh data before every test.
- `tearDown` performs cleanup after every test.
- A suite runs all related tests together.

**Next:** compare this approach with the class-based cart in D124.